# 02 · Silver — Validación de calidad físico-química

**Proyecto:** PoC Arquitectura Lakehouse para Monitoreo Ambiental Hídrico  
**Autora:** Kathiana Murillo García  
**Máster:** UNIR — Análisis y Visualización de Datos Masivos (2026–2027)  

## Objetivo
Aplicar 7 reglas de validación físico-química sobre los registros de la capa Bronze
y clasificar cada registro en `VALID`, `NULL_KEY` o `OUT_OF_RANGE`.

## Reglas de calidad implementadas
| Regla | Variable | Condición VALID | Referencia |
|-------|----------|-----------------|------------|
| R1 | Estación (ST.) | No nula | Protocolo NIES |
| R2 | Fecha (Date) | No nula | Protocolo NIES |
| R3 | DO0m | 0–20 mg/L | Wetzel (2001) |
| R4 | pH0m | 5–10 | Wetzel (2001) |
| R5 | WT0m | 0–40 °C | Wetzel (2001) |
| R6 | T-P | 0–500 µg/L | OECD (2008) |
| R7 | Chl-a | 0–300 µg/L | OECD (2008) |

## 1. Configuración

In [0]:
# Celda 1 — Parámetros
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, lit, count, round as spark_round

CATALOG = "training"
SCHEMA  = "kasumigaura"
TABLE_BRONZE = f"{CATALOG}.{SCHEMA}.bronze_water_quality"
TABLE_SILVER = f"{CATALOG}.{SCHEMA}.silver_water_quality"

# Intentar detectar catálogo activo
try:
    active_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
    CATALOG = active_catalog
    TABLE_BRONZE = f"{CATALOG}.{SCHEMA}.bronze_water_quality"
    TABLE_SILVER = f"{CATALOG}.{SCHEMA}.silver_water_quality"
except Exception:
    pass

print(f"Bronze: {TABLE_BRONZE}")
print(f"Silver: {TABLE_SILVER}")

Bronze: workspace.kasumigaura.bronze_water_quality
Silver: workspace.kasumigaura.silver_water_quality


## 2. Lectura de Bronze y definición de reglas

In [0]:
# Celda 2 — Leer capa Bronze
df_bronze = spark.table(TABLE_BRONZE)
n_bronze = df_bronze.count()
print(f"Registros Bronze: {n_bronze:,}")
df_bronze.printSchema()

Registros Bronze: 6,220
root
 |-- order: long (nullable = true)
 |-- Date: timestamp (nullable = true)
 |-- ST: long (nullable = true)
 |-- PO4_P: double (nullable = true)
 |-- DTP: double (nullable = true)
 |-- T_P: double (nullable = true)
 |-- NH4_N: double (nullable = true)
 |-- NO3+NO2_N: double (nullable = true)
 |-- NO2_N: double (nullable = true)
 |-- NO3_N: double (nullable = true)
 |-- DON: double (nullable = true)
 |-- DTN: double (nullable = true)
 |-- T_N: double (nullable = true)
 |-- D_COD: double (nullable = true)
 |-- T_COD: double (nullable = true)
 |-- Chl_a: double (nullable = true)
 |-- Pyco: double (nullable = true)
 |-- Phaeo: double (nullable = true)
 |-- SS: double (nullable = true)
 |-- POC: double (nullable = true)
 |-- PON: double (nullable = true)
 |-- C_N: double (nullable = true)
 |-- DOC: double (nullable = true)
 |-- Het_B: double (nullable = true)
 |-- TB: double (nullable = true)
 |-- GP: double (nullable = true)
 |-- NP: double (nullable = true)
 |--

In [0]:
# Celda 3 — Aplicar reglas de calidad físico-química
from pyspark.sql import Window
from pyspark.sql.functions import row_number

df_silver = (
    df_bronze
    .withColumn("rule_ok_R1", col("ST").isNotNull())
    .withColumn("rule_ok_R2", col("Date").isNotNull())
    .withColumn("rule_ok_R3",
        when(col("DO0m").isNull(), True)
        .otherwise(col("DO0m").between(0, 28)))
    .withColumn("rule_ok_R4",
        when(col("pH0m").isNull(), True)
        .otherwise(col("pH0m").between(6.0, 11.0)))
    .withColumn("rule_ok_R5",
        when(col("WT0m").isNull(), True)
        .otherwise(col("WT0m").between(0, 35)))
    .withColumn("rule_ok_R6",
        when(col("T_P").isNull(), True)
        .otherwise(col("T_P").between(0, 1300)))
    .withColumn("rule_ok_R7",
        when(col("T_N").isNull(), True)
        .otherwise(col("T_N").between(0, 5000)))
    .withColumn("rule_ok_R8",
        when(col("Transp_Secchi_depth").isNull(), True)
        .otherwise(col("Transp_Secchi_depth") > 0))
    .withColumn("rule_ok_R9",
        when(col("Chl_a").isNull(), True)
        .otherwise(col("Chl_a").between(0, 500)))
    .withColumn("all_range_ok",
        col("rule_ok_R3") & col("rule_ok_R4") & col("rule_ok_R5") &
        col("rule_ok_R6") & col("rule_ok_R7") & col("rule_ok_R8") & col("rule_ok_R9"))
    .withColumn("dq_flag",
        when(~col("rule_ok_R1") | ~col("rule_ok_R2"), "NULL_KEY")
        .when(~col("all_range_ok"), "ANOMALY")
        .otherwise("VALID"))
    .drop("rule_ok_R1","rule_ok_R2","rule_ok_R3","rule_ok_R4",
          "rule_ok_R5","rule_ok_R6","rule_ok_R7","rule_ok_R8","rule_ok_R9","all_range_ok")
)

# R10: Duplicados ST + Date
w_dup = Window.partitionBy("ST", "Date").orderBy("ST", "Date")
df_silver = (
    df_silver
    .withColumn("_row_num", row_number().over(w_dup))
    .withColumn("dq_flag",
        when((col("_row_num") > 1) & (col("dq_flag") == "VALID"), "DUPLICATE")
        .otherwise(col("dq_flag")))
    .drop("_row_num")
)

print("Vista previa:")
df_silver.select("ST", "Date", "DO0m", "pH0m", "dq_flag").show(10)

Vista previa:
+---+-------------------+----+----+-------+
| ST|               Date|DO0m|pH0m|dq_flag|
+---+-------------------+----+----+-------+
|  1|1977-04-18 00:00:00|11.0|NULL|  VALID|
|  1|1977-05-17 00:00:00|NULL|NULL|  VALID|
|  1|1977-06-29 00:00:00| 9.2|NULL|  VALID|
|  1|1977-07-25 00:00:00|11.7|NULL|  VALID|
|  1|1977-08-25 00:00:00|NULL|NULL|  VALID|
|  1|1977-09-06 00:00:00|15.0|NULL|  VALID|
|  1|1977-10-05 00:00:00| 8.0|NULL|  VALID|
|  1|1977-11-07 00:00:00| 9.0|NULL|  VALID|
|  1|1977-12-06 00:00:00|11.7|NULL|  VALID|
|  1|1978-01-20 00:00:00| 7.2|NULL|  VALID|
+---+-------------------+----+----+-------+
only showing top 10 rows


## 3. Escritura en Delta Lake (capa Silver)

In [0]:
# Celda 4 — Escribir capa Silver
(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_SILVER)
)

n_silver = spark.table(TABLE_SILVER).count()
print(f"✓ Tabla Silver escrita: {TABLE_SILVER}")
print(f"  Registros: {n_silver:,}")

✓ Tabla Silver escrita: workspace.kasumigaura.silver_water_quality
  Registros: 6,220


## 4. Análisis de calidad

In [0]:
# Celda 5 — Distribución del dq_flag
df_flag = (
    spark.table(TABLE_SILVER)
    .groupBy("dq_flag")
    .agg(
        count("*").alias("n_registros"),
        spark_round(count("*") * 100.0 / n_silver, 1).alias("porcentaje")
    )
    .orderBy("dq_flag")
)
df_flag.show()

# Verificar: n_silver == n_bronze (conservación de registros)
n_bronze_check = spark.table(TABLE_BRONZE).count()
assert n_silver == n_bronze_check, (
    f"Pérdida de registros: Bronze={n_bronze_check}, Silver={n_silver}"
)
print(f"✓ Conservación verificada: {n_silver:,} registros en Silver == {n_bronze_check:,} en Bronze")

+-------+-----------+----------+
|dq_flag|n_registros|porcentaje|
+-------+-----------+----------+
|ANOMALY|          2|       0.0|
|  VALID|       6218|     100.0|
+-------+-----------+----------+

✓ Conservación verificada: 6,220 registros en Silver == 6,220 en Bronze


In [0]:
# Celda 6 — Registros VALID por período temporal
from pyspark.sql.functions import year

print("Registros VALID por período temporal:")
(
    spark.table(TABLE_SILVER)
    .filter(col("dq_flag") == "VALID")
    .withColumn("period",
        when(year("Date") < 1984, "1977–1983")
        .when(year("Date") < 2000, "1984–1999")
        .otherwise("2000–2024")
    )
    .groupBy("period")
    .agg(count("*").alias("n_valid"))
    .orderBy("period")
    .show()
)

n_valid = spark.table(TABLE_SILVER).filter(col("dq_flag") == "VALID").count()
print(f"\n✅ Notebook 02 completado — Capa Silver: {TABLE_SILVER}")
print(f"   Registros VALID  : {n_valid:,} ({n_valid/n_silver*100:.1f}%)")
print(f"   Registros NULL_KEY: {n_silver - n_valid:,}")
print(f"   Próximo paso: ejecutar 03_gold_indicadores.ipynb")

Registros VALID por período temporal:
+---------+-------+
|   period|n_valid|
+---------+-------+
|1977–1983|   1289|
|1984–1999|   2019|
|2000–2024|   2910|
+---------+-------+


✅ Notebook 02 completado — Capa Silver: workspace.kasumigaura.silver_water_quality
   Registros VALID  : 6,218 (100.0%)
   Registros NULL_KEY: 2
   Próximo paso: ejecutar 03_gold_indicadores.ipynb
